# 📱 Telecom Customer Churn Prediction & Feature Explainability Engine
**Data Science Portfolio Framework — Predictive Modeling & Operational Customer Segmentation**

This notebook implements an automated, production-grade machine learning classification pipeline designed to proactively predict subscriber churn risk. By optimizing decision boundary thresholds via cost-sensitive learning weights, the framework balances class imbalances to secure high sensitivity on at-risk subscribers, parsing outputs directly into actionable business segments (*At Risk, Loyal, Dormant*).

In [ ]:
# Cell 1: Environment Dependencies & Configuration
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set clean visualization parameters
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 🔄 Data Ingestion & Storage Engineering
We automatically capture the official IBM Telco Portfolio from the public source repo and mirror it locally to secure reproducibility across validation pipelines.

In [ ]:
# Cell 2: Automated Dataset Extraction Hook
print("=" * 60)
print("       TELECOM CUSTOMER CHURN MACHINE LEARNING PIPELINE       ")
print("=" * 60)

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
if not os.path.exists('data'):
    os.makedirs('data')
    
print("🔄 Downloading and staging official IBM Telco customer account ledger...")
df = pd.read_csv(url)
df.to_csv('data/Telco-Customer-Churn.csv', index=False)

### 🧹 Schema Preprocessing & Categorical Feature Vectorization
This section handles blank spacing values inside the continuous financial records (`TotalCharges`) via median calculations and vectorizes string variables into structured indicators using binary one-hot encoding layouts.

In [ ]:
# Cell 3: Feature Engineering
print("🧹 Cleaning data fields and handling null gaps...")

# Address empty string anomalies found in the TotalCharges tracking variable
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = df['TotalCharges'].astype(float)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Target label mapping processing: Yes (Left) -> 1, No (Stayed) -> 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 2}).map({1: 1, 2: 0}) 

# Split out training identifiers from target indicators
X_raw = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']
X = pd.get_dummies(X_raw, drop_first=True)

# Stratified Data Split Sequence
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 🤖 Imbalance-Aware Model Selection & Classifier Compilation
We introduce cost-sensitive parameters (`class_weight='balanced_subsample'`) to force the underlying Random Forest splits to heavily weight the critical, sparse churn events, boosting model sensitivity (recall).

In [ ]:
# Cell 4: Model Fitting & Operational Performance Verification
print("🤖 Adjusting Random Forest weights for class imbalance...")
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, class_weight='balanced_subsample')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\n--- [UPGRADED MODEL EVALUATION LOG] ---")
print(f"✓ New Test Accuracy: {accuracy * 100:.2f}%")
print("\n✓ Optimized Classification Metrics:")
print(classification_report(y_test, y_pred))

### 📊 Visual Asset Generation (Diagnostic Performance & Gini Rankings)
Here we generate and export your core visual portfolio pieces—the Confusion Matrix heatmap and the Explainable AI feature importance charts—saving them directly to your `outputs/` workspace.

In [ ]:
# Cell 5: Diagnostic Visual Generation Engine
if not os.path.exists('outputs'):
    os.makedirs('outputs')
    
print("📊 Compiling Confusion Matrix Visual asset...")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Retained (0)', 'Churned (1)'],
            yticklabels=['Retained (0)', 'Churned (1)'])
plt.title('Telecom Churn Prediction Confusion Matrix', fontsize=12, weight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('outputs/churn_confusion_matrix.png', dpi=300)
plt.show()
plt.close()

print("📊 Extracting Global Model Explanations...")
importances = model.feature_importances_
indices = np.argsort(importances)[::-1][:10]
plt.figure(figsize=(11, 6))
sns.barplot(x=importances[indices], y=X.columns[indices], palette="viridis", hue=X.columns[indices], legend=False)
plt.title("Top 10 Operational Drivers of Customer Churn Risk", fontsize=14, weight='bold')
plt.xlabel("Global Model Contribution Importance Score (Gini)")
plt.ylabel("Telemetry Features")
plt.tight_layout()
plt.savefig('outputs/churn_feature_importance.png', dpi=300)
plt.show()
plt.close()

### 🎯 Strategic Account Segmentation & Pipeline Export
Instead of flat binary states, we extract raw internal prediction probabilities to rank, route, and partition subscribers into concrete operational groups (`At Risk`, `Loyal`, `Dormant`), caching the output to a clean manifest table.

In [ ]:
# Cell 6: Customer Segmentation Engine
print("🎯 Categorizing subscriber accounts into strategic business segments...")

probabilities = model.predict_proba(X_test)[:, 1]
segment_df = pd.DataFrame({
    'true_status': y_test.values,
    'churn_probability': probabilities,
    'tenure_months': X_test['tenure'] if 'tenure' in X_test.columns else 0
})

conditions = [
    (segment_df['churn_probability'] >= 0.60),
    (segment_df['churn_probability'] < 0.30) & (segment_df['tenure_months'] >= 24),
    (segment_df['churn_probability'] < 0.60) & (segment_df['tenure_months'] < 6)
]
choices = ['At Risk', 'Loyal', 'Dormant']
segment_df['customer_segment'] = np.select(conditions, choices, default='Standard Active')

segment_summary = segment_df['customer_segment'].value_counts()
segment_df.to_csv('outputs/customer_segments_manifest.csv', index=False)

print("\n--- [CUSTOMER SEGMENTATION SUMMARY] ---")
for segment, count in segment_summary.items():
    print(f"📦 Segment Group: {segment:<15} | Volume: {count} accounts mapped.")
    
print("✓ Outputs saved successfully to 'outputs/' directory!")
print("=" * 60)